# CA6127 — remaining experiments on Colab

**Before you start**
1. Upload `ca6127_colab.zip` to the root of **My Drive**.
2. Runtime → Change runtime type → **GPU**. Best: **A100** (everything, fastest). **L4** also runs FLUX.1 Kontext (4-bit).
   **T4** runs everything *except* Kontext.
3. For Kontext only: accept the licence at huggingface.co/black-forest-labs/FLUX.1-Kontext-dev, create a **Read** token,
   and add it in the key icon (Secrets) on the left as `HF_TOKEN` with *Notebook access* on. Never paste the token into a cell.

Run the cells top to bottom. Everything is written inside `My Drive/CA6127` and every step is resumable:
after a disconnect, run all cells again and it continues where it stopped.

| Step | A100 time (approx.) |
|---|---|
| dev phase: select the hyper-parameters | 5 min |
| test phase (SD1.5, hybrid, InstructPix2Pix) | 30 min |
| SDXL, 160 images | 30 min |
| FLUX.1 Kontext, dev sweep + test | 60 min |
| speed benchmark + final evaluation | 30 min |

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
RUN_KONTEXT = vram >= 20
print(f'{name}, {vram:.0f} GB VRAM ->', 'Kontext: bf16' if vram >= 38 else 'Kontext: 4-bit' if RUN_KONTEXT else 'Kontext skipped (needs >= 20 GB)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
PROJECT = '/content/drive/MyDrive/CA6127'
ZIP = '/content/drive/MyDrive/ca6127_colab.zip'
if not os.path.exists(f'{PROJECT}/src/run_all.py'):
    print('unzipping (first run only) ...')
    zipfile.ZipFile(ZIP).extractall(PROJECT)
os.chdir(PROJECT)
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['SDXL_LIMIT'] = '160'
print(os.getcwd())

In [ ]:
# Colab already ships torch / torchvision / opencv / scikit-image / pandas / matplotlib
!pip install -q diffusers transformers accelerate lpips easyocr qrcode bitsandbytes sentencepiece protobuf
!pip install -q --no-deps simple-lama-inpainting

In [ ]:
# Kontext only: check the HF token and the licence (401 = token missing, 403 = licence not accepted)
if RUN_KONTEXT:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import whoami, hf_hub_download
    print('logged in as', whoami()['name'])
    hf_hub_download('black-forest-labs/FLUX.1-Kontext-dev', 'model_index.json')
    print('Kontext access OK')

In [ ]:
# weights go to the Colab disk (not Drive); after a runtime restart they are downloaded again (~5-10 min)
!python src/download_models.py {'--kontext' if RUN_KONTEXT else ''}

## 1. Dev phase
Selects the hyper-parameters on dev and writes `results/selected_hparams.json`.
All other dev runs were finished on the Mac and are skipped.

In [ ]:
!python src/run_all.py --phase dev
!cat results/selected_hparams.json

## 2. Test phase (ground-truth masks)

In [ ]:
!python src/run_all.py --phase test

## 3. SDXL-Inpaint on all 160 test images

In [ ]:
!python src/run_all.py --phase sdxl

## 4. FLUX.1 Kontext [dev] (guidance sweep on dev, then test)

In [ ]:
if RUN_KONTEXT:
    !python src/run_all.py --phase kontext
else:
    print('skipped: GPU has < 20 GB')

## 5. Speed benchmark (every method on this one GPU, same 10 images)

In [ ]:
!python src/benchmark_speed.py

## 6. Final evaluation
Scores **every** run again on this machine so all numbers come from the same OCR / QR / LPIPS setup.

In [ ]:
!rm -rf results/_cache/clean_ocr
!python src/evaluate.py --force

## 7. Bring the results back to the Mac
Download `My Drive/results_colab.zip`, then on the Mac:
```
unzip -o ~/Downloads/results_colab.zip -d /path/to/this/repo
```

In [ ]:
!cd {PROJECT} && rm -f /content/drive/MyDrive/results_colab.zip && zip -qr /content/drive/MyDrive/results_colab.zip results -x "results/_cache/*"
!ls -lh /content/drive/MyDrive/results_colab.zip